# Challenge 06 -- Agent Harness Fundamentals: From Agent to Production Agent (Coach Solution)

Use this notebook with Student/Challenge-06.md.

Scenario: You are building an AI Workshop Assistant that helps a Microsoft PSA prepare customer workshops.

Core idea: **Agent = Model + Harness**

## Section 0 -- Introduction

A model can generate text.
An agent can take actions.
An agent harness provides runtime capabilities for reliable production behavior.



In [ ]:
import os
import time
from dataclasses import dataclass

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential, ChainedTokenCredential, InteractiveBrowserCredential

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
TENANT_ID = os.getenv("AZURE_TENANT_ID")

openai_client = None
REAL_LLM_READY = False

if PROJECT_ENDPOINT and MODEL_DEPLOYMENT_NAME:
    try:
        credential = ChainedTokenCredential(
            AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
            InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
        )
        project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
        openai_client = project_client.get_openai_client()
        REAL_LLM_READY = True
        print(f"Real LLM ready: {MODEL_DEPLOYMENT_NAME}")
    except Exception as ex:
        print(f"Real LLM setup skipped: {ex}")
else:
    print("Real LLM setup skipped: set AZURE_AI_FOUNDRY_ENDPOINT and AZURE_OPENAI_DEPLOYMENT_NAME in .env")

def fake_chat_completion(prompt: str) -> str:
    return f"Model response to: {prompt}"

def real_chat_completion(prompt: str) -> str:
    if not REAL_LLM_READY:
        return "[Real model unavailable in this session.]"
    try:
        response = openai_client.responses.create(
            model=MODEL_DEPLOYMENT_NAME,
            input=prompt,
        )
        return response.output_text
    except Exception as ex:
        return f"[Real model call failed: {ex}]"

def search_workshop_assets(topic: str) -> str:
    assets = {
        "azure ai foundry workshop": "Slides: Foundry-Intro, Demo: Agent Playground, Lab: Tool Calling",
        "ai discovery cards": "Cards: Persona Mapping, Use-Case Prioritization, Value Framing",
        "manufacturing": "Case Study: Predictive Maintenance, Demo: Defect Detection"
    }
    return assets.get(topic.lower(), "No specific assets found.")

def summarize_context(messages):
    if not messages:
        return "Summary: (empty)"
    head = messages[:2]
    tail = messages[-2:] if len(messages) > 2 else []
    return "Summary: " + " | ".join(head + tail)

def estimate_tokens(text: str) -> int:
    return max(1, len(text.split()))

## Section 1 -- A Model Is Not an Agent

In [ ]:
prompt = "Prepare a customer workshop outline for Contoso."
real_model_only_response = real_chat_completion(prompt)
simulated_model_only_response = fake_chat_completion(prompt)

print("Real model-only response:")
print(real_model_only_response)
print("\nSimulated baseline response:")
print(simulated_model_only_response)

print("\nCoach note: model-only may look polished, but it has no guaranteed tool execution, memory persistence, workflow control, approvals, or observability.")

## A/B Comparison -- Model Only vs Harness-Enabled

Use the same prompt across both paths so students can see practical differences in output structure and execution traceability.

In [ ]:
def run_minimal_harness(user_prompt: str):
    plan = [
        "Research customer",
        "Identify AI opportunities",
        "Create workshop agenda",
    ]
    tool_result = search_workshop_assets("Azure AI Foundry workshop")
    memory_hint = "Preferred examples: Manufacturing use cases"
    final_answer = (
        f"Prompt: {user_prompt}\n"
        f"Plan: {plan}\n"
        f"Tool result: {tool_result}\n"
        f"Memory hint: {memory_hint}\n"
        "Execution summary: tool_used=True, approval_required=False"
    )
    return {
        "plan": plan,
        "tool_result": tool_result,
        "memory_hint": memory_hint,
        "final_answer": final_answer,
    }

comparison_prompt = "Prepare a one-day Azure AI Foundry workshop for Fabrikam."
model_only = real_chat_completion(comparison_prompt)
harness_result = run_minimal_harness(comparison_prompt)

print("Model-only output:\n")
print(model_only)
print("\n" + "=" * 60 + "\n")
print("Harness-enabled output:\n")
print(harness_result["final_answer"])

## Section 2 -- Build Your First Agent Loop

In [ ]:
context = ["User asks for workshop assets on Azure AI Foundry"]

@dataclass
class LoopResponse:
    requires_tool: bool
    tool_name: str
    tool_input: str
    final_answer: str

def agent_step(ctx):
    latest = ctx[-1].lower()
    if "assets" in latest and "tool_result:" not in latest:
        return LoopResponse(True, "search_workshop_assets", "Azure AI Foundry workshop", "")
    return LoopResponse(False, "", "", "Final answer prepared using tool output in context.")

response = agent_step(context)

if response.requires_tool:
    tool_result = f"tool_result: {search_workshop_assets(response.tool_input)}"
    context.append(tool_result)

# Continue loop once after tool execution
response = agent_step(context)
print("Context:", context)
print("Final:", response.final_answer)

## Section 3 -- Tools

In [ ]:
tools = [
    search_workshop_assets
]

test_prompt = "Find assets for an Azure AI Foundry workshop."
tool_output = tools[0]("Azure AI Foundry workshop")
final_answer = f"Prompt: {test_prompt}\nTool output: {tool_output}"
print(final_answer)

## Section 4 -- Memory

In [ ]:
memory_store = {}

def save_memory(key, value):
    memory_store[key] = value

def load_memory(key):
    return memory_store.get(key, "No saved memory for this key.")

save_memory("workshop_preferences", "My preferred workshop examples are Manufacturing use cases.")
print("What workshop examples do I prefer?")
print(load_memory("workshop_preferences"))

## Section 5 -- Context Management

In [ ]:
LIMIT = 35
context = [
    "User: Build an executive workshop agenda.",
    "Assistant: Drafted a 1-day agenda skeleton.",
    "User: Add manufacturing examples and architecture options.",
    "Assistant: Added examples, diagrams, and implementation notes.",
    "User: Include follow-up tasks and role assignments."
]

token_count = sum(estimate_tokens(msg) for msg in context)
print(f"Before token count: {token_count}")

if token_count > LIMIT:
    summary = summarize_context(context)
    context = [summary]

after_count = sum(estimate_tokens(msg) for msg in context)
print(f"After token count: {after_count}")
print(context[0])

## Section 6 -- Planning

In [ ]:
todo_list = []
todo_list.append("Research customer")
todo_list.append("Identify AI opportunities")
todo_list.append("Create agenda")
todo_list.append("Generate workshop preparation guide")

print("Generated plan:")
for i, item in enumerate(todo_list, start=1):
    print(f"{i}. {item}")

## Section 7 -- Sub-agents

In [ ]:
def ResearchAgent(customer):
    return f"Research for {customer}: business priorities include modernization, copilots, and governance"

def AgendaAgent(research):
    return "Agenda: Intro to Foundry, Discovery Cards, Agent Build Lab, Governance Wrap-up"

def SummaryAgent(research, agenda):
    return f"Summary ready.\n- {research}\n- {agenda}"

customer = "Contoso"
research_results = ResearchAgent(customer)
agenda = AgendaAgent(research_results)
final_output = SummaryAgent(research_results, agenda)

print(final_output)

## Section 8 -- Human in the Loop

In [ ]:
def stop():
    return "Action stopped by approval gate."

def run_hitl_path(approved):
    action = "send_email"
    if action == "send_email":
        if not approved:
            return stop()
        return "Email sent."
    return "No gated action requested."

print("Denied path:", run_hitl_path(False))
print("Approved path:", run_hitl_path(True))

## Section 9 -- Observability

In [ ]:
events = []

def log_event(name, payload):
    events.append({"event": name, "payload": payload, "timestamp": time.time()})

start_time = time.time()
log_event("tool_call", {"tool": "search_workshop_assets", "input": "Azure AI Foundry workshop"})
tool_payload = search_workshop_assets("Azure AI Foundry workshop")
log_event("tool_result", {"result": tool_payload})

tokens_used = estimate_tokens(tool_payload) + estimate_tokens(final_output)
log_event("token_count", {"tokens": tokens_used})

step_latency = 0.032
log_event("latency", {"seconds": step_latency})

time.sleep(0.05)
duration = time.time() - start_time
log_event("run_duration", {"seconds": duration})

tools_used = len([e for e in events if e["event"] == "tool_call"])
estimated_cost = round(tokens_used * 0.000002, 6)

print("Agent Run Summary")
print("-------------------")
print(f"Tools Used: {tools_used}")
print(f"Tokens: {tokens_used}")
print(f"Duration: {duration:.4f} seconds")
print(f"Estimated Cost: ${estimated_cost}")

## Section 10 -- Capstone

In [ ]:
def build_plan(customer):
    return [
        f"Research {customer}",
        "Identify AI opportunities",
        "Create agenda",
        "Generate workshop preparation guide"
    ]

def recommended_demos():
    return [
        "Foundry Agent Playground walkthrough",
        "Tool-calling with workshop asset lookup",
        "Observability dashboard review"
    ]

def architecture_recommendations():
    return [
        "Use hosted agent runtime for production endpoints",
        "Add approval gates for outbound actions",
        "Emit traces and metrics for every run"
    ]

def follow_up_actions():
    return [
        "Share workshop recap within 24 hours",
        "Schedule technical deep-dive",
        "Define pilot success metrics"
    ]

capstone_input = "Prepare a one-day Azure AI Foundry workshop for Fabrikam."
customer = "Fabrikam"

plan = build_plan(customer)
research = ResearchAgent(customer)
agenda = AgendaAgent(research)
demos = recommended_demos()
arch = architecture_recommendations()
actions = follow_up_actions()

approved = True
approval_result = "Approved" if approved else "Denied"

execution_summary = {
    "tools_used": tools_used,
    "tokens": tokens_used,
    "duration_seconds": round(duration, 4),
    "estimated_cost": estimated_cost,
    "approval": approval_result
}

print("Input:", capstone_input)
print("\nResearch summary:")
print(research)
print("\nWorkshop agenda:")
print(agenda)
print("\nRecommended demos:")
for d in demos:
    print("-", d)
print("\nArchitecture recommendations:")
for a in arch:
    print("-", a)
print("\nFollow-up actions:")
for f in actions:
    print("-", f)
print("\nExecution summary:")
print(execution_summary)

## Final Reflection (Sample)

| Capability | Model Only | Agent Harness |
|------------|-----------|---------------|
| Tool Use | No deterministic execution path | Registers and executes tools |
| Memory | No persistent state between calls | Save/load memory across runs |
| Planning | One-shot text | Explicit actionable plan |
| Context Management | Unbounded until model limit | Summarization/compaction strategy |
| Sub-Agents | No orchestration role split | Coordinator delegates to specialists |
| Human Approval | No governance gate | Approval checkpoint for sensitive actions |
| Observability | Minimal visibility | Step-level metrics, traces, and summaries |

Why a harness is required:
A production agent must do more than generate text. It must execute tools safely, manage memory and context over time, plan and coordinate tasks, include human control points, and emit telemetry for reliability and governance. Those are harness responsibilities, not model-only capabilities.

## Bonus Challenge -- Reusable AgentHarness Class (Completed)

In [ ]:
class AgentHarness:
    def __init__(self):
        self.tools = {}
        self.memory = {}
        self.metrics = {"tool_calls": 0, "tokens": 0, "runs": 0}

    def register_tool(self, name, fn):
        self.tools[name] = fn

    def save_memory(self, key, value):
        self.memory[key] = value

    def create_plan(self, customer):
        return [
            f"Research {customer}",
            "Identify AI opportunities",
            "Create agenda",
            "Generate workshop preparation guide"
        ]

    def request_approval(self, action):
        # Simulated approval gate for lab repeatability.
        return action != "send_email_blocked"

    def run(self, prompt):
        started = time.time()
        self.metrics["runs"] += 1

        customer = "Fabrikam" if "Fabrikam" in prompt else "Contoso"
        plan = self.create_plan(customer)

        tool_name = "search_workshop_assets"
        tool_result = self.tools[tool_name]("Azure AI Foundry workshop") if tool_name in self.tools else "Tool unavailable"
        self.metrics["tool_calls"] += 1

        self.save_memory("last_customer", customer)
        self.metrics["tokens"] += estimate_tokens(prompt) + estimate_tokens(tool_result)

        approved = self.request_approval("send_email")
        duration = round(time.time() - started, 4)

        return {
            "plan": plan,
            "tool_result": tool_result,
            "memory": dict(self.memory),
            "approved": approved,
            "metrics": {**self.metrics, "duration_seconds": duration}
        }

h = AgentHarness()
h.register_tool("search_workshop_assets", search_workshop_assets)
bonus_result = h.run("Prepare a one-day Azure AI Foundry workshop for Fabrikam.")
print(bonus_result)